[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/ONNX_Tutorial/blob/main/01_ONNX_with_Python/03_Initializers_and_Attributes/Initializers_and_Attributes_Apply.ipynb)

# 1.3 Initializers and Attributes — Hands-On Practice

Embed weights as **initializers** inside the model graph and control operator behaviour through **attributes**.

---

## Table of Contents

| # | Section | Focus |
|---|---------|-------|
| 1 | [Setup & Imports](#section-1) | Dependencies |
| 2 | [Exercise 1: Model with Initializers](#section-2) | Embed weights so only X is a runtime input |
| 3 | [Exercise 2: Transpose via Attribute](#section-3) | Use the `perm` attribute on `Transpose` |
| 4 | [Exercise 3: Conv2D Attributes](#section-4) | Kernel, stride, padding attributes |
| 5 | [Exercise 4: Initializer vs Input](#section-5) | Inspect the difference programmatically |
| 6 | [Exercise 5: Replace Initializer Values](#section-6) | Hot-swap weights in a saved model |
| 7 | [Exercise 6: Multi-Attribute Operators](#section-7) | Build a full Conv → BN → Relu block |
| 8 | [Exercise 7: File-Size Impact](#section-8) | Measure how initializer size affects .onnx |
| 9 | [Exercise 8: Visualize Weight Distributions](#section-9) | Histogram of embedded weights |
| 10 | [Challenge: 2-Layer MLP with Embedded Weights](#section-10) | Full self-contained model |
| 11 | [Summary](#section-11) | Recap |

<a id='section-1'></a>
## Section 1: Setup & Imports

In [ ]:
# Uncomment for Colab:
# !pip install onnx onnxruntime matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt
import os
import time
import copy

from onnx import TensorProto, load, save
from onnx.helper import (
    make_model, make_node, make_graph,
    make_tensor_value_info, make_opsetid)
from onnx.checker import check_model
from onnx import numpy_helper
from onnx.numpy_helper import from_array, to_array
import onnxruntime as ort

print(f'ONNX Runtime version: {ort.__version__}')
print('Setup complete!')

<a id='section-2'></a>
## Section 2: Exercise 1 — Build a Model with Initializers

### Why Initializers?

Without initializers, **every** tensor the graph needs must be fed at runtime.
Initializers let us *embed* weight tensors inside the `.onnx` file so the model
is self-contained — exactly like a `.pt` or `.h5` file.

### Mathematical formulation

We build $Y = XW + b$ where $W \in \mathbb{R}^{2 \times 2}$ and $b \in \mathbb{R}^{2}$
are stored as initializers. Only $X \in \mathbb{R}^{N \times 2}$ is a runtime input.

```
Runtime input:   X (N×2)
                  │
                  ▼
              ┌──────┐
  Initializer │  W   │── MatMul ──► XW
              └──────┘              │
              ┌──────┐              ▼
  Initializer │  b   │── Add ────► Y
              └──────┘
```

In [ ]:
# Fixed weights stored inside the model
W_data = np.array([[0.5, -0.3], [0.2, 0.8]], dtype=np.float32)
b_data = np.array([0.1, -0.1], dtype=np.float32)

W_init = from_array(W_data, name='W')
b_init = from_array(b_data, name='b')

# Only X is a true runtime input
X = make_tensor_value_info('X', TensorProto.FLOAT, [None, 2])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None, 2])

graph = make_graph(
    [make_node('MatMul', ['X', 'W'], ['XW']),
     make_node('Add', ['XW', 'b'], ['Y'])],
    'model_with_init',
    inputs=[X],          # note: only X — W and b come from initializers
    outputs=[Y],
    initializer=[W_init, b_init])

model_init = make_model(graph, opset_imports=[make_opsetid('', 17)])
check_model(model_init)

# Run — we only feed X
sess = ort.InferenceSession(
    model_init.SerializeToString(),
    providers=['CPUExecutionProvider'])

x = np.array([[1, 2], [3, 4], [5, 6]], dtype=np.float32)
result = sess.run(None, {'X': x})[0]
expected = x @ W_data + b_data

print('ONNX result:\n', result)
print('NumPy check:\n', expected)
print(f'Match: {np.allclose(result, expected)}')
print(f'\nModel inputs (runtime): {[i.name for i in model_init.graph.input]}')
print(f'Initializers (embedded): {[i.name for i in model_init.graph.initializer]}')

<a id='section-3'></a>
## Section 3: Exercise 2 — Transpose via the `perm` Attribute

### What are attributes?

Attributes are **compile-time constants** attached to operator nodes.
Unlike inputs (which carry data at runtime), attributes control *how*
an operator behaves and are fixed once the graph is built.

The `Transpose` operator uses the `perm` attribute to specify the
permutation order.  For a 2-D matrix, `perm=[1, 0]` flips rows and columns:

$$A^T_{ij} = A_{ji}$$

We compute $Y = X \cdot A^T + B$ by inserting a `Transpose` node with
`perm=[1, 0]` before the `MatMul`.

In [ ]:
X = make_tensor_value_info('X', TensorProto.FLOAT, [None, None])
A = make_tensor_value_info('A', TensorProto.FLOAT, [None, None])
B = make_tensor_value_info('B', TensorProto.FLOAT, [None, None])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None])

# The Transpose node uses the perm attribute
transpose_node = make_node('Transpose', ['A'], ['tA'], perm=[1, 0])

graph = make_graph(
    [transpose_node,
     make_node('MatMul', ['X', 'tA'], ['XA']),
     make_node('Add', ['XA', 'B'], ['Y'])],
    'transpose_model', [X, A, B], [Y])
model_t = make_model(graph, opset_imports=[make_opsetid('', 17)])
check_model(model_t)

# Inspect the perm attribute
for node in model_t.graph.node:
    for attr in node.attribute:
        print(f'  Node {node.op_type}: attr "{attr.name}" = {list(attr.ints)}')

# Run inference
sess_t = ort.InferenceSession(
    model_t.SerializeToString(), providers=['CPUExecutionProvider'])

x = np.array([[1, 2, 3]], dtype=np.float32)           # (1, 3)
a = np.array([[0.5, -0.3, 0.1],
              [0.2,  0.8, -0.5]], dtype=np.float32)    # (2, 3)  → A^T is (3, 2)
b = np.array([[0.1, -0.1]], dtype=np.float32)          # (1, 2)

result = sess_t.run(None, {'X': x, 'A': a, 'B': b})[0]
expected = x @ a.T + b

print(f'\nONNX result: {result}')
print(f'NumPy X@A.T+B: {expected}')
print(f'Match: {np.allclose(result, expected)}')

<a id='section-4'></a>
## Section 4: Exercise 3 — Conv2D Attributes

The `Conv` operator is a great example of attribute-heavy nodes. It requires:

| Attribute | Type | Meaning |
|-----------|------|--------|
| `kernel_shape` | `[int]` | Spatial size of the convolution filter |
| `strides` | `[int]` | Step size for sliding the kernel |
| `pads` | `[int]` | Zero-padding before/after each spatial axis |
| `group` | `int` | Number of groups for grouped convolution |

The output spatial dimension is:

$$H_{\text{out}} = \left\lfloor \frac{H_{\text{in}} + \text{pad}_{\text{top}} + \text{pad}_{\text{bot}} - k_H}{s_H} \right\rfloor + 1$$

In [ ]:
np.random.seed(42)

# Conv weights: (out_channels=8, in_channels=3, kH=3, kW=3)
W_conv = from_array(
    np.random.randn(8, 3, 3, 3).astype(np.float32) * 0.1, name='W')
B_conv = from_array(
    np.zeros(8, dtype=np.float32), name='B')

X = make_tensor_value_info('X', TensorProto.FLOAT, [1, 3, 16, 16])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [1, 8, 16, 16])

conv_node = make_node(
    'Conv', ['X', 'W', 'B'], ['Y'],
    kernel_shape=[3, 3],
    strides=[1, 1],
    pads=[1, 1, 1, 1],   # same padding
    group=1)

graph = make_graph(
    [conv_node], 'conv_demo', [X], [Y], initializer=[W_conv, B_conv])
model_conv = make_model(graph, opset_imports=[make_opsetid('', 17)])
check_model(model_conv)

# Inspect all attributes
print('Conv node attributes:')
for attr in model_conv.graph.node[0].attribute:
    val = list(attr.ints) if attr.ints else attr.i
    print(f'  {attr.name}: {val}')

# Run
sess_conv = ort.InferenceSession(
    model_conv.SerializeToString(), providers=['CPUExecutionProvider'])
x = np.random.randn(1, 3, 16, 16).astype(np.float32)
result = sess_conv.run(None, {'X': x})[0]

print(f'\nInput shape:  {x.shape}')
print(f'Output shape: {result.shape}')
assert result.shape == (1, 8, 16, 16), 'Same-padding should preserve spatial dims'
print('Assertion passed: spatial dims preserved with same-padding')

<a id='section-5'></a>
## Section 5: Exercise 4 — Initializer vs Input: Programmatic Inspection

A common source of confusion is that in some ONNX versions, initializers
also appear in `graph.input`.  Here we inspect both lists and show how
to distinguish **true runtime inputs** from embedded weights.

```
graph.input        = {X, W, b}   ← everything the graph consumes
graph.initializer  = {W, b}      ← values embedded in the file

runtime inputs     = graph.input − graph.initializer = {X}
```

In [ ]:
def inspect_inputs_vs_initializers(model):
    """Classify tensor names as runtime-input, initializer, or both."""
    init_names = {i.name for i in model.graph.initializer}
    input_names = {i.name for i in model.graph.input}
    output_names = {o.name for o in model.graph.output}

    runtime_inputs = input_names - init_names

    print(f'graph.input names:       {sorted(input_names)}')
    print(f'graph.initializer names: {sorted(init_names)}')
    print(f'graph.output names:      {sorted(output_names)}')
    print(f'\nTrue runtime inputs:     {sorted(runtime_inputs)}')
    print(f'Embedded weights:        {sorted(init_names)}')
    return runtime_inputs, init_names


print('=== Model with initializers ===')
rt, wt = inspect_inputs_vs_initializers(model_init)
assert rt == {'X'}, 'Only X should be a runtime input'
assert 'W' in wt and 'b' in wt

print('\n=== Model without initializers (all runtime) ===')
X2 = make_tensor_value_info('X', TensorProto.FLOAT, [None, None])
A2 = make_tensor_value_info('A', TensorProto.FLOAT, [None, None])
Y2 = make_tensor_value_info('Y', TensorProto.FLOAT, [None])
g2 = make_graph(
    [make_node('MatMul', ['X', 'A'], ['Y'])], 'no_init', [X2, A2], [Y2])
m2 = make_model(g2, opset_imports=[make_opsetid('', 17)])
inspect_inputs_vs_initializers(m2)

<a id='section-6'></a>
## Section 6: Exercise 5 — Replace Initializer Values (Weight Hot-Swap)

A powerful pattern: load a model, **swap** its weight tensors with new values
(e.g., from a fine-tuned checkpoint), then save the updated model.

This is the ONNX equivalent of `model.load_state_dict(new_weights)` in PyTorch.

In [ ]:
# Save the original model
save(model_init, 'original.onnx')

# Load, swap weights, save
loaded = load('original.onnx')

new_W = np.array([[1.0, 0.0], [0.0, 1.0]], dtype=np.float32)  # identity
new_b = np.array([0.0, 0.0], dtype=np.float32)                # zero bias

for i, init in enumerate(loaded.graph.initializer):
    if init.name == 'W':
        loaded.graph.initializer[i].CopyFrom(from_array(new_W, name='W'))
        print(f'Replaced W with identity matrix')
    elif init.name == 'b':
        loaded.graph.initializer[i].CopyFrom(from_array(new_b, name='b'))
        print(f'Replaced b with zeros')

check_model(loaded)
save(loaded, 'swapped.onnx')

# Verify: with identity W and zero b, Y should equal X
sess_swap = ort.InferenceSession('swapped.onnx', providers=['CPUExecutionProvider'])
x_test = np.array([[7.0, -3.0]], dtype=np.float32)
result_swap = sess_swap.run(None, {'X': x_test})[0]

print(f'\nInput:  {x_test}')
print(f'Output: {result_swap}')
assert np.allclose(result_swap, x_test), 'With W=I and b=0, output should equal input'
print('Assertion passed: Y = X (identity transform)')

<a id='section-7'></a>
## Section 7: Exercise 6 — Multi-Attribute Operators (Conv → BN → Relu)

Build a full convolutional block with three operators, each using different attributes.
The `BatchNormalization` operator has the `epsilon` attribute (and `momentum` for training).

$$\text{BN}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

where $\gamma$ (scale), $\beta$ (bias), $\mu$ (mean), $\sigma^2$ (variance) are initializers
and $\epsilon$ is an attribute.

In [ ]:
np.random.seed(0)
C = 8  # channels

# Conv initializers
W_c = from_array(np.random.randn(C, 3, 3, 3).astype(np.float32) * 0.1, name='conv_W')
B_c = from_array(np.zeros(C, dtype=np.float32), name='conv_B')

# BN initializers (pretend these came from training)
scale = from_array(np.ones(C, dtype=np.float32), name='bn_scale')
bias  = from_array(np.zeros(C, dtype=np.float32), name='bn_bias')
mean  = from_array(np.zeros(C, dtype=np.float32), name='bn_mean')
var   = from_array(np.ones(C, dtype=np.float32), name='bn_var')

X = make_tensor_value_info('X', TensorProto.FLOAT, [1, 3, 16, 16])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, [1, C, 16, 16])

nodes = [
    make_node('Conv', ['X', 'conv_W', 'conv_B'], ['conv_out'],
             kernel_shape=[3, 3], pads=[1, 1, 1, 1]),
    make_node('BatchNormalization',
             ['conv_out', 'bn_scale', 'bn_bias', 'bn_mean', 'bn_var'],
             ['bn_out'], epsilon=1e-5),
    make_node('Relu', ['bn_out'], ['Y']),
]

graph = make_graph(
    nodes, 'conv_bn_relu', [X], [Y],
    initializer=[W_c, B_c, scale, bias, mean, var])
model_cbr = make_model(graph, opset_imports=[make_opsetid('', 17)])
check_model(model_cbr)

# Report all attributes across all nodes
print('Operator attributes in Conv→BN→Relu block:')
for node in model_cbr.graph.node:
    attrs = []
    for a in node.attribute:
        if a.ints:
            attrs.append(f'{a.name}={list(a.ints)}')
        elif a.type == 1:  # FLOAT
            attrs.append(f'{a.name}={a.f}')
        elif a.type == 2:  # INT
            attrs.append(f'{a.name}={a.i}')
    attr_str = ', '.join(attrs) if attrs else '(none)'
    print(f'  {node.op_type:25s} attrs: {attr_str}')

# Run
sess_cbr = ort.InferenceSession(
    model_cbr.SerializeToString(), providers=['CPUExecutionProvider'])
x_in = np.random.randn(1, 3, 16, 16).astype(np.float32)
y_out = sess_cbr.run(None, {'X': x_in})[0]

print(f'\nInput:  {x_in.shape}')
print(f'Output: {y_out.shape}')
print(f'All >= 0 (Relu): {(y_out >= 0).all()}')
print(f'Total initializers: {len(model_cbr.graph.initializer)}')

<a id='section-8'></a>
## Section 8: Exercise 7 — File-Size Impact of Initializers

Initializers dominate model file size.  For $N$ float32 parameters the
lower bound is $4N$ bytes.  Let's measure empirically how model size
scales with the weight matrix dimensions.

In [ ]:
configs = [(4, 4), (16, 16), (64, 64), (128, 128), (256, 256), (512, 512)]
sizes = []
param_counts = []

for rows, cols in configs:
    W = from_array(np.random.randn(rows, cols).astype(np.float32), name='W')
    b = from_array(np.random.randn(cols).astype(np.float32), name='b')
    _X = make_tensor_value_info('X', TensorProto.FLOAT, [None, rows])
    _Y = make_tensor_value_info('Y', TensorProto.FLOAT, [None, cols])
    _g = make_graph(
        [make_node('MatMul', ['X', 'W'], ['XW']),
         make_node('Add', ['XW', 'b'], ['Y'])],
        'size_test', [_X], [_Y], [W, b])
    _m = make_model(_g, opset_imports=[make_opsetid('', 17)])
    sz = len(_m.SerializeToString())
    n_params = rows * cols + cols
    sizes.append(sz)
    param_counts.append(n_params)
    overhead = (sz - 4 * n_params) / (4 * n_params) * 100
    print(f'  W[{rows:>4}x{cols:>4}]  params={n_params:>8,}  '
          f'file={sz:>10,} B  overhead={overhead:.1f}%')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(param_counts, [s / 1024 for s in sizes], 'o-',
        color='#2E86C1', linewidth=2, markersize=8, label='Actual .onnx')
ax.plot(param_counts, [p * 4 / 1024 for p in param_counts], 's--',
        color='#E74C3C', linewidth=2, markersize=6, label='Theoretical min (4N bytes)')
ax.set_xlabel('Number of Parameters', fontsize=12)
ax.set_ylabel('Size (KB)', fontsize=12)
ax.set_title('ONNX File Size vs Number of Initializer Parameters',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<a id='section-9'></a>
## Section 9: Exercise 8 — Visualize Weight Distributions

When debugging a model, checking the *distribution* of embedded weights
catches common issues:

- All zeros → forgot to load the checkpoint
- Very large values → exploding gradients during training
- Bimodal → possible quantization artifact

In [ ]:
def plot_initializer_distributions(model, max_inits=6):
    """Histogram each float initializer in the model."""
    float_inits = []
    for init in model.graph.initializer:
        arr = to_array(init)
        if arr.dtype in (np.float32, np.float64, np.float16):
            float_inits.append((init.name, arr))
    if not float_inits:
        print('No float initializers found.')
        return
    float_inits = float_inits[:max_inits]
    n = len(float_inits)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]
    for ax, (name, arr) in zip(axes, float_inits):
        flat = arr.ravel()
        ax.hist(flat, bins=50, color='#3498DB', alpha=0.8, edgecolor='white')
        ax.axvline(x=0, color='red', linestyle='--', alpha=0.5)
        ax.set_title(f'{name} {list(arr.shape)}', fontsize=10, fontweight='bold')
        ax.set_xlabel('Value')
        ax.set_ylabel('Count')
        ax.text(0.98, 0.95,
               f'$\\mu$={flat.mean():.4f}\n$\\sigma$={flat.std():.4f}',
               transform=ax.transAxes, ha='right', va='top', fontsize=8,
               bbox=dict(boxstyle='round', fc='white', alpha=0.8))
    plt.suptitle('Initializer Weight Distributions', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

plot_initializer_distributions(model_cbr)

<a id='section-10'></a>
## Section 10: Challenge — 2-Layer MLP with Embedded Weights

Build a fully self-contained 2-layer MLP:

$$H = \text{Relu}(X W_1 + b_1)$$
$$Y = H W_2 + b_2$$

where **all** four weight tensors ($W_1, b_1, W_2, b_2$) are embedded as
initializers.  Only $X$ is a runtime input.

```
X (N×4) ──► MatMul(·,W1) ──► Add(·,b1) ──► Relu ──► MatMul(·,W2) ──► Add(·,b2) ──► Y (N×2)
              ▲                  ▲                      ▲                  ▲
         init W1 (4×8)      init b1 (8)           init W2 (8×2)      init b2 (2)
```

In [ ]:
np.random.seed(123)

# Xavier-initialised weights
W1_data = (np.random.randn(4, 8) * np.sqrt(2.0 / 4)).astype(np.float32)
b1_data = np.zeros(8, dtype=np.float32)
W2_data = (np.random.randn(8, 2) * np.sqrt(2.0 / 8)).astype(np.float32)
b2_data = np.zeros(2, dtype=np.float32)

inits = [
    from_array(W1_data, 'W1'), from_array(b1_data, 'b1'),
    from_array(W2_data, 'W2'), from_array(b2_data, 'b2'),
]

X = make_tensor_value_info('X', TensorProto.FLOAT, ['N', 4])
Y = make_tensor_value_info('Y', TensorProto.FLOAT, ['N', 2])

nodes = [
    make_node('MatMul', ['X', 'W1'], ['XW1']),
    make_node('Add',    ['XW1', 'b1'], ['pre_relu']),
    make_node('Relu',   ['pre_relu'], ['H']),
    make_node('MatMul', ['H', 'W2'], ['HW2']),
    make_node('Add',    ['HW2', 'b2'], ['Y']),
]

graph = make_graph(nodes, 'mlp_self_contained', [X], [Y], initializer=inits)
model_mlp = make_model(graph, opset_imports=[make_opsetid('', 17)])
check_model(model_mlp)

# Save, load, run — fully self-contained
save(model_mlp, 'mlp_standalone.onnx')
sess_mlp = ort.InferenceSession('mlp_standalone.onnx',
                                 providers=['CPUExecutionProvider'])

x_test = np.random.randn(20, 4).astype(np.float32)
y_onnx = sess_mlp.run(None, {'X': x_test})[0]

# NumPy reference
h_np = np.maximum(0, x_test @ W1_data + b1_data)
y_np = h_np @ W2_data + b2_data

print(f'Input shape:  {x_test.shape}')
print(f'Output shape: {y_onnx.shape}')
print(f'Match: {np.allclose(y_onnx, y_np, atol=1e-6)}')
print(f'\nModel summary:')
print(f'  Runtime inputs:   {[i.name for i in model_mlp.graph.input]}')
print(f'  Initializers:     {[i.name for i in model_mlp.graph.initializer]}')
print(f'  Nodes:            {[n.op_type for n in model_mlp.graph.node]}')
print(f'  File size:        {os.path.getsize("mlp_standalone.onnx")} bytes')
total_params = sum(to_array(i).size for i in model_mlp.graph.initializer)
print(f'  Total parameters: {total_params}')

In [ ]:
# Performance measurement
batch_sizes = [1, 4, 16, 64, 256, 1024]
latencies = []

for bs in batch_sizes:
    x_perf = np.random.randn(bs, 4).astype(np.float32)
    sess_mlp.run(None, {'X': x_perf})  # warmup
    times = []
    for _ in range(200):
        t0 = time.perf_counter()
        sess_mlp.run(None, {'X': x_perf})
        times.append((time.perf_counter() - t0) * 1e6)
    avg = np.mean(times)
    latencies.append(avg)
    print(f'  batch={bs:5d}  latency={avg:8.1f} us')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(batch_sizes, latencies, 'o-', color='#27AE60', linewidth=2, markersize=8)
ax.set_xlabel('Batch Size', fontsize=12)
ax.set_ylabel('Latency (us)', fontsize=12)
ax.set_title('Self-Contained MLP: Inference Latency vs Batch Size',
             fontsize=13, fontweight='bold')
ax.set_xscale('log', base=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Cleanup
for f in ['original.onnx', 'swapped.onnx', 'mlp_standalone.onnx']:
    if os.path.exists(f):
        os.remove(f)
print('Cleanup complete!')

<a id='section-11'></a>

---

## Summary

| Exercise | Skill | Key Concept |
|----------|-------|-------------|
| 1 | Build model with initializers | Embed $W, b$ so only $X$ is runtime |
| 2 | Transpose attribute | `perm=[1,0]` controls axis reordering |
| 3 | Conv2D attributes | `kernel_shape`, `strides`, `pads` |
| 4 | Initializer vs Input | `graph.input - graph.initializer` = runtime |
| 5 | Hot-swap weights | Replace initializers in a saved model |
| 6 | Multi-attribute block | Conv → BN ($\epsilon$) → Relu |
| 7 | File-size analysis | Initializers dominate `.onnx` size |
| 8 | Weight distributions | Histogram for debugging |
| Challenge | Self-contained MLP | All weights embedded, benchmark |

**Key insight:** Initializers make ONNX models *self-contained* — a single `.onnx`
file holds both the computation graph and all trained weights.  Attributes
control operator *behaviour* at graph-construction time.

**Next:** [Opset and Metadata](../04_Opset_and_Metadata/) — Version your models and enrich them with metadata.